# Music Annotation Project V8.7
## Notebook 49: Residual-Mask Oracle-Gap Decomposition

**Runtime:** CPU. Select **Runtime > Run all**.

This is a **development-only, read-only diagnostic**. It tests whether the observable M3 residual-mask state contains enough stable signal to justify a learned M5 challenger.

### Non-negotiable contract
- no model training, threshold optimisation or restorer changes;
- M3 observable candidates and features are built and semantically locked before clean truth is loaded;
- clean truth is used only after the lock to assign diagnostic classes and evaluate separability;
- intervention remains inside the authoritative mask;
- raw page identity, source-group identity and x/y coordinates remain identifiers, never predictors;
- any roster, hash, shape, lock or truth-firewall failure invalidates the run;
- outputs are development evidence, not production authorisation.

### Diagnostic classes
- **PROTECT**: trustworthy observed material corresponding to genuine print;
- **REMOVE**: annotation/non-print or paper material that does not require structural reconstruction;
- **RECONSTRUCT**: genuine print exists but is unsupported by trustworthy observed bytes;
- **UNCERTAIN**: truth assignment is unreliable under the frozen tolerance and ambiguity rules.

The notebook supports two source layouts:
1. a durable Notebook 45 run containing `region_manifest.parquet` plus per-page NPZ arrays; or
2. a fail-closed adapter assembled from the Notebook 45 run and the project image/mask records.

If the exact Notebook 45 paths differ, edit only the clearly marked configuration block.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib scikit-learn scipy


## 1. Configuration and declared inputs


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, os, platform, random, shutil, sys, zipfile
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
NB45_ROOT = ROOT / 'residual_mask_readiness_v8_6'
OUTPUT_ROOT = ROOT / 'residual_mask_oracle_gap_v8_7'

# Set this to one exact completed Notebook 45 run when known.
SOURCE_NB45_RUN_OVERRIDE = None

# Optional exact paths. Leave None to use source records or controlled discovery.
DEVELOPMENT_MANIFEST_OVERRIDE = None
OBSERVED_DIR_OVERRIDE = None
CLEAN_DIR_OVERRIDE = None
V82_PREDICTIONS_OVERRIDE = None
V84_GROUP_PREDICTIONS_OVERRIDE = None

# Truth-independent analysis constants.
SEED = 42
MIN_REGION_AREA = 1
CONTEXT_RING_RADIUS = 18
BOUNDARY_BAND_RADIUS = 2
EDGE_LOW = 70
EDGE_HIGH = 170
DARK_THRESHOLD = 200
PRINT_INK_TOLERANCE = 1
REGISTRATION_UNCERTAINTY_RADIUS = 2
MAX_VISUAL_CASES_PER_QUEUE = 40
DIAGNOSTIC_CV_MAX_FOLDS = 10

# Diagnostic-only labels. These are frozen before outcomes are inspected.
# Observed dark + clean print nearby -> PROTECT
# No clean print nearby -> REMOVE
# Clean print nearby but observed is not trustworthy dark material -> RECONSTRUCT
# Clean-print boundary / excluded border / inconsistent evidence -> UNCERTAIN
LABEL_VERSION = 'n49-v1-protect-remove-reconstruct-uncertain'

RUN = datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook49_oracle_gap'
OUT = OUTPUT_ROOT / RUN
INPUTS = OUT / '01_inputs'
LOCK = OUT / '02_observable_lock'
TRUTH = OUT / '03_truth_unlock'
TABLES = OUT / '04_tables'
VISUAL = OUT / '05_visual_audit'
AUDIT = OUT / '06_audit'
UPLOAD = OUT / 'upload'
for p in [OUT, INPUTS, LOCK, TRUTH, TABLES, VISUAL, AUDIT, UPLOAD]:
    p.mkdir(parents=True, exist_ok=False)

random.seed(SEED)
np.random.seed(SEED)
print('Output:', OUT)


## 2. Helpers and complete-run resolution


In [ ]:
SUPPORTED_IMAGES={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):
            h.update(b)
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a)
    h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes())
    return h.hexdigest()

def atomic_json(path,value):
    path=Path(path); tmp=Path(str(path)+'.tmp')
    tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str))
    tmp.replace(path)

def image_map(folder):
    folder=Path(folder)
    return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED_IMAGES} if folder.exists() else {}

def load_npz(path):
    with np.load(path,allow_pickle=False) as z:
        return {k:z[k] for k in z.files}

def semantic_frame_hash(df, columns):
    q=df.loc[:,columns].copy().sort_values(columns,kind='mergesort').reset_index(drop=True)
    return hashlib.sha256(pd.util.hash_pandas_object(q,index=False).values.tobytes()).hexdigest()

def require(condition,message):
    if not condition:
        failure={'created_utc':datetime.now(timezone.utc).isoformat(),'message':message}
        atomic_json(AUDIT/'fatal_failure.json',failure)
        raise RuntimeError(message)

def find_complete_nb45_runs(root):
    required_any=[
        ['mask_lock.json','region_manifest.parquet'],
        ['mask_lock.json','tables/mask_readiness_metrics.csv'],
        ['mask_lock.json','oracle_gap_summary.csv'],
    ]
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if not p.is_dir(): continue
            if any(all((p/x).exists() for x in req) for req in required_any): hits.append(p)
    return sorted(set(hits),key=lambda p:p.stat().st_mtime)

runs=find_complete_nb45_runs(NB45_ROOT)
SOURCE=Path(SOURCE_NB45_RUN_OVERRIDE) if SOURCE_NB45_RUN_OVERRIDE else (runs[-1] if runs else None)
require(SOURCE is not None, f'No complete Notebook 45 run found below {NB45_ROOT}. Set SOURCE_NB45_RUN_OVERRIDE.')
print('Notebook 45 source:',SOURCE)

source_files=[]
for p in sorted(SOURCE.rglob('*')):
    if p.is_file(): source_files.append({'relative_path':str(p.relative_to(SOURCE)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(source_files).to_csv(AUDIT/'source_nb45_inventory.csv',index=False)
atomic_json(OUT/'source_record.json',{'source_nb45_run':str(SOURCE),'files':len(source_files),'inventory_sha256':sha256_file(AUDIT/'source_nb45_inventory.csv')})


## 3. Resolve the Notebook 45 data contract

Notebook 49 requires one row per page/source group and the following semantic arrays for every page:

- `observed`: aligned annotated greyscale page;
- `authoritative_mask`: intervention authority;
- `M3`: locked observable residual mask;
- `M4`: post-lock oracle residual mask;
- `protected_print`: observed PRINT already protected before M3 where available;
- route/support context: preserve, remove, ambiguous, horizontal, vertical and compact;
- optional saved V8 evidence such as `p_print`, V8 decision, V8.4 safety and value maps.

The resolver accepts common key aliases but stops if `observed`, `authoritative_mask`, `M3` or `M4` cannot be proven.


In [ ]:
ALIASES={
 'observed':['observed','aligned_observed','annotated','image'],
 'authoritative_mask':['authoritative_mask','removal_mask','mask','prepared_mask'],
 'M0':['M0','m0'], 'M1':['M1','m1'], 'M2':['M2','m2'], 'M3':['M3','m3'], 'M4':['M4','m4','oracle_mask','oracle_residual'],
 'protected_print':['protected_print','protected','print_protected','v84_protected'],
 'preserve':['preserve','g80_preserve'], 'remove':['remove','g80_remove'], 'ambiguous':['ambiguous','g80_ambiguous'],
 'horizontal':['horizontal','g80_horizontal'], 'vertical':['vertical','g80_vertical'], 'compact':['compact','g80_compact'],
 'paper':['paper','background_surface'], 'normalised':['normalised','normalized','norm'],
 'p_print':['p_print','v82_p_print'], 'v84_safety':['v84_safety','safety_probability'], 'v84_value':['v84_value','value_probability'],
}

def alias_value(data,key,required=False):
    for name in ALIASES[key]:
        if name in data: return data[name]
    if required: raise KeyError(f'Missing required semantic array {key}; aliases={ALIASES[key]}')
    return None

def candidate_npz_files(source):
    return sorted([p for p in Path(source).rglob('*.npz') if 'checkpoint' not in p.name.lower()])

# Prefer an explicit page-array manifest if Notebook 45 wrote one.
manifest_candidates=[SOURCE/'region_manifest.parquet',SOURCE/'tables/region_manifest.parquet',SOURCE/'page_manifest.csv',SOURCE/'tables/page_manifest.csv']
manifest_path=next((p for p in manifest_candidates if p.exists()),None)
manifest45=None
if manifest_path:
    manifest45=pd.read_parquet(manifest_path) if manifest_path.suffix=='.parquet' else pd.read_csv(manifest_path)
    print('Notebook 45 manifest:',manifest_path,'rows:',len(manifest45))

npz_files=candidate_npz_files(SOURCE)
npz_by_stem={p.stem:p for p in npz_files}
print('Candidate page NPZ files:',len(npz_files))

# Development page/source manifest.
manifest_options=[]
if DEVELOPMENT_MANIFEST_OVERRIDE: manifest_options.append(Path(DEVELOPMENT_MANIFEST_OVERRIDE))
manifest_options += [SOURCE/'inputs/development_score_manifest.csv',SOURCE/'development_score_manifest.csv',ROOT/'development_score_manifest.csv']
dev_manifest_path=next((p for p in manifest_options if p.exists()),None)
require(dev_manifest_path is not None,'No development_score_manifest.csv found. Set DEVELOPMENT_MANIFEST_OVERRIDE.')
dev_manifest=pd.read_csv(dev_manifest_path,dtype=str)
require({'page','source_group'}.issubset(dev_manifest.columns),'Development manifest requires page and source_group.')
require(not dev_manifest.page.duplicated().any(),'Development manifest page values must be unique.')
dev_manifest['page']=dev_manifest.page.astype(str); dev_manifest['source_group']=dev_manifest.source_group.astype(str)
shutil.copy2(dev_manifest_path,INPUTS/'development_score_manifest.csv')
meta=dev_manifest.set_index('page')

# Optional fallback image directories.
obs_dirs=[Path(OBSERVED_DIR_OVERRIDE)] if OBSERVED_DIR_OVERRIDE else [ROOT/'aligned',ROOT/'source'/'annotated']
clean_dirs=[Path(CLEAN_DIR_OVERRIDE)] if CLEAN_DIR_OVERRIDE else [ROOT/'source'/'clean']
OBS_MAP={}
for d in obs_dirs:
    OBS_MAP.update({k:v for k,v in image_map(d).items() if k not in OBS_MAP})
CLEAN_MAP={}
for d in clean_dirs:
    CLEAN_MAP.update({k:v for k,v in image_map(d).items() if k not in CLEAN_MAP})

def resolve_page_npz(page):
    direct=[SOURCE/'page_arrays'/f'{page}.npz',SOURCE/'masks'/f'{page}.npz',SOURCE/'arrays'/f'{page}.npz',SOURCE/f'{page}.npz']
    for p in direct:
        if p.exists(): return p
    if page in npz_by_stem: return npz_by_stem[page]
    matches=[p for p in npz_files if page in p.stem]
    if len(matches)==1: return matches[0]
    return None

pages=[]
if manifest45 is not None and 'page' in manifest45.columns:
    pages=sorted(set(manifest45.page.astype(str)) & set(dev_manifest.page))
else:
    pages=sorted(set(dev_manifest.page) & set(npz_by_stem))
require(bool(pages),'No Notebook 45 pages could be resolved from manifest/NPZ records.')
print('Resolved pages:',len(pages),'source groups:',dev_manifest[dev_manifest.page.isin(pages)].source_group.nunique())


## 4. Build observable page arrays and candidate population without opening clean truth


In [ ]:
def bool_array(a,shape=None):
    if a is None: return None
    q=np.asarray(a)
    if shape is not None and q.shape!=shape: raise ValueError(f'Shape mismatch {q.shape} != {shape}')
    return q>0

def background_surface(observed,mask,sigma=45.0):
    visible=(~mask)&(observed>=200)
    values=observed.astype(np.float32)*visible
    weights=visible.astype(np.float32)
    num=cv2.GaussianBlur(values,(0,0),sigma); den=cv2.GaussianBlur(weights,(0,0),sigma)
    fallback=cv2.GaussianBlur(observed,(0,0),sigma)
    return np.clip(np.where(den>0.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)

def ring(mask,radius):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*radius+1,2*radius+1))
    return (cv2.dilate(mask.astype(np.uint8),k)>0)&~mask

def local_fraction(a,k=5):
    return cv2.blur(a.astype(np.float32),(k,k))

def distance_to(mask):
    # Distance to True set. Empty mask returns a large finite field.
    if not mask.any(): return np.full(mask.shape,float(max(mask.shape)),np.float32)
    return cv2.distanceTransform((~mask).astype(np.uint8),cv2.DIST_L2,5)

def endpoint_maps(binary):
    b=binary.astype(np.uint8)
    neighbours=cv2.filter2D(b,-1,np.ones((3,3),np.uint8))-b
    return b.astype(bool)&(neighbours==1), b.astype(bool)&(neighbours>=3)

page_runtime={}; page_audit=[]
for page in pages:
    npz=resolve_page_npz(page)
    require(npz is not None,f'{page}: no unique Notebook 45 page NPZ resolved.')
    data=load_npz(npz)
    observed=alias_value(data,'observed')
    if observed is None:
        require(page in OBS_MAP,f'{page}: observed array absent from NPZ and no fallback image found.')
        observed=cv2.imread(str(OBS_MAP[page]),0)
    observed=np.asarray(observed)
    require(observed.ndim==2,f'{page}: observed page must be greyscale 2D.')
    shape=observed.shape
    authority=bool_array(alias_value(data,'authoritative_mask',True),shape)
    m3=bool_array(alias_value(data,'M3',True),shape)
    m4=bool_array(alias_value(data,'M4',True),shape)
    require(np.all(~m3|authority),f'{page}: M3 exceeds authoritative mask.')
    require(np.all(~m4|authority),f'{page}: M4 exceeds authoritative mask.')
    protected=bool_array(alias_value(data,'protected_print'),shape)
    if protected is None: protected=np.zeros(shape,bool)
    arrays={}
    for k in ['preserve','remove','ambiguous','horizontal','vertical','compact']:
        q=bool_array(alias_value(data,k),shape)
        arrays[k]=q if q is not None else np.zeros(shape,bool)
    paper=alias_value(data,'paper')
    if paper is None: paper=background_surface(observed,authority)
    paper=np.asarray(paper,dtype=np.uint8)
    norm=alias_value(data,'normalised')
    if norm is None: norm=cv2.divide(observed,np.maximum(paper,1),scale=255)
    norm=np.asarray(norm,dtype=np.uint8)
    edge=cv2.Canny(observed,EDGE_LOW,EDGE_HIGH)>0
    depth=cv2.distanceTransform(m3.astype(np.uint8),cv2.DIST_L2,5)
    m3_boundary=m3 & ~cv2.erode(m3.astype(np.uint8),np.ones((3,3),np.uint8)).astype(bool)
    boundary_distance=cv2.distanceTransform((~m3_boundary).astype(np.uint8),cv2.DIST_L2,5)
    endpoints,junctions=endpoint_maps((observed<DARK_THRESHOLD)&~m3)
    p_print=alias_value(data,'p_print'); v84_safety=alias_value(data,'v84_safety'); v84_value=alias_value(data,'v84_value')
    for name,val in [('p_print',p_print),('v84_safety',v84_safety),('v84_value',v84_value)]:
        if val is not None and np.asarray(val).shape!=shape: raise RuntimeError(f'{page}: {name} shape mismatch')
    page_runtime[page]={'npz':npz,'observed':observed,'authority':authority,'m3':m3,'m4':m4,'protected':protected,'paper':paper,'norm':norm,'edge':edge,'depth':depth,'boundary_distance':boundary_distance,'endpoints':endpoints,'junctions':junctions,'p_print':p_print,'v84_safety':v84_safety,'v84_value':v84_value,**arrays}
    page_audit.append({'page':page,'source_group':meta.loc[page,'source_group'],'npz':str(npz),'npz_sha256':sha256_file(npz),'height':shape[0],'width':shape[1],'authority_pixels':int(authority.sum()),'m3_pixels':int(m3.sum()),'m4_pixels':int(m4.sum()),'m3_inside_authority':True,'m4_inside_authority':True})

page_audit_df=pd.DataFrame(page_audit)
page_audit_df.to_csv(AUDIT/'page_array_audit.csv',index=False)
print(page_audit_df[['authority_pixels','m3_pixels','m4_pixels']].sum())


## 5. Observable pixel and region feature construction


In [ ]:
PIXEL_FEATURES=[
 'observed_value','normalised_value','paper_value','mask_depth','boundary_distance','distance_to_protected',
 'edge','local_edge_fraction','local_dark_fraction','preserve','remove','ambiguous','horizontal','vertical','compact',
 'local_preserve_fraction','local_remove_fraction','local_ambiguous_fraction','local_horizontal_fraction','local_vertical_fraction','local_compact_fraction',
 'endpoint_near_fraction','junction_near_fraction','p_print','v84_safety','v84_value'
]

pixel_rows=[]; region_rows=[]; region_masks={}
for page,z in page_runtime.items():
    observed=z['observed']; m3=z['m3']; ys,xs=np.where(m3)
    if len(ys)==0: continue
    local={
      'edge':local_fraction(z['edge']), 'dark':local_fraction(observed<DARK_THRESHOLD),
      'preserve':local_fraction(z['preserve']), 'remove':local_fraction(z['remove']), 'ambiguous':local_fraction(z['ambiguous']),
      'horizontal':local_fraction(z['horizontal']), 'vertical':local_fraction(z['vertical']), 'compact':local_fraction(z['compact']),
      'endpoint':local_fraction(z['endpoints'],7), 'junction':local_fraction(z['junctions'],7),
    }
    dist_protected=distance_to(z['protected'])
    def optional_at(a): return np.asarray(a)[ys,xs] if a is not None else np.full(len(ys),np.nan)
    frame=pd.DataFrame({
      'source_group':str(meta.loc[page,'source_group']),'page':page,'y':ys.astype(np.int32),'x':xs.astype(np.int32),
      'observed_value':observed[ys,xs].astype(np.int16),'normalised_value':z['norm'][ys,xs].astype(np.int16),'paper_value':z['paper'][ys,xs].astype(np.int16),
      'mask_depth':z['depth'][ys,xs].astype(np.float32),'boundary_distance':z['boundary_distance'][ys,xs].astype(np.float32),'distance_to_protected':dist_protected[ys,xs].astype(np.float32),
      'edge':z['edge'][ys,xs].astype(np.uint8),'local_edge_fraction':local['edge'][ys,xs],'local_dark_fraction':local['dark'][ys,xs],
      'preserve':z['preserve'][ys,xs].astype(np.uint8),'remove':z['remove'][ys,xs].astype(np.uint8),'ambiguous':z['ambiguous'][ys,xs].astype(np.uint8),
      'horizontal':z['horizontal'][ys,xs].astype(np.uint8),'vertical':z['vertical'][ys,xs].astype(np.uint8),'compact':z['compact'][ys,xs].astype(np.uint8),
      'local_preserve_fraction':local['preserve'][ys,xs],'local_remove_fraction':local['remove'][ys,xs],'local_ambiguous_fraction':local['ambiguous'][ys,xs],
      'local_horizontal_fraction':local['horizontal'][ys,xs],'local_vertical_fraction':local['vertical'][ys,xs],'local_compact_fraction':local['compact'][ys,xs],
      'endpoint_near_fraction':local['endpoint'][ys,xs],'junction_near_fraction':local['junction'][ys,xs],
      'p_print':optional_at(z['p_print']),'v84_safety':optional_at(z['v84_safety']),'v84_value':optional_at(z['v84_value']),
    })
    n,labels,stats,_=cv2.connectedComponentsWithStats(m3.astype(np.uint8),8)
    frame['region_id']=[f'{page}:r{int(labels[y,x])}' for y,x in zip(ys,xs)]
    pixel_rows.append(frame)
    for cid in range(1,n):
        comp=labels==cid; area=int(comp.sum())
        if area<MIN_REGION_AREA: continue
        cy,cx=np.where(comp); ring_mask=ring(comp,CONTEXT_RING_RADIUS)
        boundary=comp & ~cv2.erode(comp.astype(np.uint8),np.ones((3,3),np.uint8)).astype(bool)
        region_id=f'{page}:r{cid}'; region_masks[region_id]=(page,comp)
        region_rows.append({
          'source_group':str(meta.loc[page,'source_group']),'page':page,'region_id':region_id,'component':cid,
          'x':int(stats[cid,cv2.CC_STAT_LEFT]),'y':int(stats[cid,cv2.CC_STAT_TOP]),'w':int(stats[cid,cv2.CC_STAT_WIDTH]),'h':int(stats[cid,cv2.CC_STAT_HEIGHT]),'area':area,
          'aspect_ratio':float(max(stats[cid,2],stats[cid,3])/max(min(stats[cid,2],stats[cid,3]),1)),'fill_ratio':float(area/max(stats[cid,2]*stats[cid,3],1)),
          'maximum_mask_depth':float(z['depth'][comp].max()),'mean_mask_depth':float(z['depth'][comp].mean()),
          'boundary_pixels':int(boundary.sum()),'trusted_print_boundary_contact':float((cv2.dilate(z['protected'].astype(np.uint8),np.ones((3,3),np.uint8))>0)[boundary].mean()) if boundary.any() else 0.0,
          'paper_boundary_contact':float((z['paper'][boundary]>=200).mean()) if boundary.any() else 0.0,
          'edge_boundary_contact':float(z['edge'][boundary].mean()) if boundary.any() else 0.0,
          'horizontal_fraction':float(z['horizontal'][comp].mean()),'vertical_fraction':float(z['vertical'][comp].mean()),'compact_fraction':float(z['compact'][comp].mean()),
          'preserve_fraction':float(z['preserve'][comp].mean()),'remove_fraction':float(z['remove'][comp].mean()),'ambiguous_fraction':float(z['ambiguous'][comp].mean()),
          'context_edge_density':float(z['edge'][ring_mask].mean()) if ring_mask.any() else 0.0,'context_dark_density':float((observed[ring_mask]<DARK_THRESHOLD).mean()) if ring_mask.any() else 0.0,
          'endpoint_count_near':int(z['endpoints'][cv2.dilate(comp.astype(np.uint8),np.ones((7,7),np.uint8))>0].sum()),'junction_count_near':int(z['junctions'][cv2.dilate(comp.astype(np.uint8),np.ones((7,7),np.uint8))>0].sum()),
          'm4_overlap_fraction_locked':float((z['m4']&comp).sum()/area),
        })

pixels=pd.concat(pixel_rows,ignore_index=True).sort_values(['source_group','page','y','x']).reset_index(drop=True)
regions=pd.DataFrame(region_rows).sort_values(['source_group','page','component']).reset_index(drop=True)
require(not pixels.empty,'Observable M3 pixel population is empty.')
require(not pixels.duplicated(['page','y','x']).any(),'Duplicate page/y/x candidate rows.')
pixels.to_parquet(LOCK/'observable_pixels.parquet',index=False)
regions.to_parquet(LOCK/'observable_regions.parquet',index=False)
print('Observable pixels:',len(pixels),'regions:',len(regions))


## 6. Write and verify the observable lock before clean truth


In [ ]:
feature_schema={
 'label_version':LABEL_VERSION,
 'pixel_identifiers':['source_group','page','y','x','region_id'],
 'pixel_predictors':PIXEL_FEATURES,
 'region_identifiers':['source_group','page','region_id','component','x','y','w','h'],
 'prohibited_predictors':['source_group','page','x','y','clean','truth','class','outcome','failure','mae','precision','recall'],
 'truth_loaded_before_lock':False,
}
atomic_json(LOCK/'feature_schema.json',feature_schema)
lock_payload={
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'source_nb45_run':str(SOURCE),
 'source_inventory_sha256':sha256_file(AUDIT/'source_nb45_inventory.csv'),
 'development_manifest_sha256':sha256_file(INPUTS/'development_score_manifest.csv'),
 'feature_schema_sha256':sha256_file(LOCK/'feature_schema.json'),
 'observable_pixels_sha256':sha256_file(LOCK/'observable_pixels.parquet'),
 'observable_regions_sha256':sha256_file(LOCK/'observable_regions.parquet'),
 'ordered_pixel_identity_sha256':semantic_frame_hash(pixels,['source_group','page','y','x','region_id']),
 'pages':int(pixels.page.nunique()),'source_groups':int(pixels.source_group.nunique()),'pixels':int(len(pixels)),'regions':int(len(regions)),
 'semantic_page_hashes':{p:{k:array_hash(v.astype(np.uint8) if v.dtype==bool else v) for k,v in page_runtime[p].items() if isinstance(v,np.ndarray)} for p in sorted(page_runtime)},
}
LOCK_ID=object_hash(lock_payload)
atomic_json(LOCK/'candidate_lock.json',{'lock_id':LOCK_ID,'payload':lock_payload,'status':'LOCKED_BEFORE_CLEAN_TRUTH'})
# Exact reload gate.
reloaded_pixels=pd.read_parquet(LOCK/'observable_pixels.parquet')
reloaded_regions=pd.read_parquet(LOCK/'observable_regions.parquet')
require(sha256_file(LOCK/'observable_pixels.parquet')==lock_payload['observable_pixels_sha256'],'Observable pixel lock hash mismatch.')
require(semantic_frame_hash(reloaded_pixels,['source_group','page','y','x','region_id'])==lock_payload['ordered_pixel_identity_sha256'],'Observable semantic identity mismatch.')
print('OBSERVABLE LOCK CREATED:',LOCK_ID)


## 7. Truth unlock and diagnostic class assignment

The following cell is the first cell permitted to open clean reference pages. The labels are diagnostic outcomes, not inference features.


In [ ]:
require((LOCK/'candidate_lock.json').exists(),'Observable lock must exist before truth unlock.')
clean_missing=[p for p in pages if p not in CLEAN_MAP]
require(not clean_missing,f'Clean truth missing for pages: {clean_missing}')

def printed_ink(im):
    return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

def truth_for_page(page):
    clean=cv2.imread(str(CLEAN_MAP[page]),0)
    require(clean is not None,f'{page}: clean page unreadable.')
    z=page_runtime[page]
    require(clean.shape==z['observed'].shape,f'{page}: clean/observed shape mismatch.')
    ink=printed_ink(clean)
    k=np.ones((2*PRINT_INK_TOLERANCE+1,2*PRINT_INK_TOLERANCE+1),np.uint8)
    near=cv2.dilate(ink.astype(np.uint8),k)>0
    eroded=cv2.erode(ink.astype(np.uint8),k)>0
    registration_band=np.logical_xor(near,eroded)
    border=np.zeros(clean.shape,bool); r=REGISTRATION_UNCERTAINTY_RADIUS
    border[:r,:]=True; border[-r:,:]=True; border[:,:r]=True; border[:,-r:]=True
    return clean,ink,near,registration_band|border

truth_meta=[]; classes=[]
for page,g in pixels.groupby('page',sort=False):
    clean,ink,near,uncertain=truth_for_page(page)
    ys=g.y.to_numpy(int); xs=g.x.to_numpy(int)
    observed=page_runtime[page]['observed'][ys,xs]
    is_uncertain=uncertain[ys,xs]
    has_print=near[ys,xs]
    trustworthy_observed_dark=observed<DARK_THRESHOLD
    label=np.full(len(g),'UNCERTAIN',object)
    label[(~is_uncertain)&has_print&trustworthy_observed_dark]='PROTECT'
    label[(~is_uncertain)&(~has_print)]='REMOVE'
    label[(~is_uncertain)&has_print&(~trustworthy_observed_dark)]='RECONSTRUCT'
    q=g[['page','y','x']].copy(); q['diagnostic_class']=label; classes.append(q)
    truth_meta.append({'page':page,'clean_path':str(CLEAN_MAP[page]),'clean_sha256':sha256_file(CLEAN_MAP[page]),'clean_ink_pixels':int(ink.sum()),'truth_uncertain_pixels':int(uncertain.sum())})
truth_labels=pd.concat(classes,ignore_index=True)
pixels=pixels.merge(truth_labels,on=['page','y','x'],validate='one_to_one')
pd.DataFrame(truth_meta).to_csv(TRUTH/'truth_input_manifest.csv',index=False)
pixels.to_parquet(TRUTH/'pixel_oracle_gap.parquet',index=False)
print(pixels.diagnostic_class.value_counts())


## 8. Region composition and oracle-gap table


In [ ]:
composition=(pixels.groupby(['source_group','page','region_id','diagnostic_class'],as_index=False).size().rename(columns={'size':'pixels'}))
wide=composition.pivot_table(index=['source_group','page','region_id'],columns='diagnostic_class',values='pixels',fill_value=0).reset_index()
for c in ['PROTECT','REMOVE','RECONSTRUCT','UNCERTAIN']:
    if c not in wide: wide[c]=0
wide['classified_pixels']=wide[['PROTECT','REMOVE','RECONSTRUCT','UNCERTAIN']].sum(axis=1)
for c in ['PROTECT','REMOVE','RECONSTRUCT','UNCERTAIN']:
    wide[c.lower()+'_fraction']=wide[c]/np.maximum(wide.classified_pixels,1)
wide['dominant_class']=wide[['PROTECT','REMOVE','RECONSTRUCT','UNCERTAIN']].idxmax(axis=1)
wide['mixed_operational_region']=(wide[['PROTECT','REMOVE','RECONSTRUCT']]>0).sum(axis=1)>1
regions=regions.merge(wide,on=['source_group','page','region_id'],how='left',validate='one_to_one')
regions.to_parquet(TRUTH/'region_oracle_gap.parquet',index=False)
regions.to_csv(TABLES/'region_oracle_gap.csv',index=False)

class_page=(pixels.groupby(['source_group','page','diagnostic_class'],as_index=False).size().rename(columns={'size':'pixels'}))
class_source=(pixels.groupby(['source_group','diagnostic_class'],as_index=False).size().rename(columns={'size':'pixels'}))
class_page.to_csv(TABLES/'class_prevalence_by_page.csv',index=False)
class_source.to_csv(TABLES/'class_prevalence_by_source_group.csv',index=False)
print('Mixed operational regions:',int(regions.mixed_operational_region.fillna(False).sum()),'of',len(regions))


## 9. Feature contrasts without fitting the M5 model


In [ ]:
FORBIDDEN_TOKENS=['clean','truth','class','outcome','failure','mae','precision','recall']
eligible_features=[c for c in PIXEL_FEATURES if c in pixels.columns and not any(t in c.lower() for t in FORBIDDEN_TOKENS)]
contrast_rows=[]
for cls in ['PROTECT','REMOVE','RECONSTRUCT']:
    target=pixels.diagnostic_class==cls
    reference=pixels.diagnostic_class=='RECONSTRUCT' if cls!='RECONSTRUCT' else pixels.diagnostic_class!='RECONSTRUCT'
    for feature in eligible_features:
        a=pd.to_numeric(pixels.loc[target,feature],errors='coerce').replace([np.inf,-np.inf],np.nan).dropna()
        b=pd.to_numeric(pixels.loc[reference,feature],errors='coerce').replace([np.inf,-np.inf],np.nan).dropna()
        if len(a)<2 or len(b)<2: continue
        # Rank-biserial effect from Mann-Whitney U.
        try:
            u,p=mannwhitneyu(a,b,alternative='two-sided'); rbc=2*u/(len(a)*len(b))-1
        except Exception:
            u,p,rbc=np.nan,np.nan,np.nan
        contrast_rows.append({'target_class':cls,'reference':'RECONSTRUCT' if cls!='RECONSTRUCT' else 'NOT_RECONSTRUCT','feature':feature,'target_n':len(a),'reference_n':len(b),'target_mean':float(a.mean()),'reference_mean':float(b.mean()),'target_median':float(a.median()),'reference_median':float(b.median()),'standardised_mean_difference':float((a.mean()-b.mean())/max(np.sqrt((a.var(ddof=1)+b.var(ddof=1))/2),1e-12)),'rank_biserial':float(rbc),'mannwhitney_p':float(p)})
contrasts=pd.DataFrame(contrast_rows)
contrasts.to_csv(TABLES/'feature_contrasts.csv',index=False)
print('Feature contrasts:',len(contrasts))


## 10. Predeclared simple separability probes

These probes are **diagnostic only**. They are not persisted as Notebook 50 models and cannot become the M5 policy. The aim is to test whether class separation survives page/source-group-held-out evaluation using a simple regularised linear model.


In [ ]:
def grouped_splits(groups):
    n=groups.nunique()
    if n<2: return []
    splitter=LeaveOneGroupOut() if n<=DIAGNOSTIC_CV_MAX_FOLDS else GroupKFold(n_splits=DIAGNOSTIC_CV_MAX_FOLDS)
    return list(splitter.split(np.zeros(len(groups)),groups=groups))

def diagnostic_probe(target_name,positive_class,negative_classes,group_column='source_group'):
    d=pixels[pixels.diagnostic_class.isin([positive_class]+negative_classes)].copy()
    y=(d.diagnostic_class==positive_class).astype(int)
    X=d[eligible_features].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    # Remove empty/constant columns inside the diagnostic design only.
    keep=[c for c in X if X[c].notna().any() and X[c].nunique(dropna=True)>1]
    X=X[keep]; groups=d[group_column].astype(str)
    out=np.full(len(d),np.nan); fold_rows=[]
    for fold,(tr,te) in enumerate(grouped_splits(groups),1):
        if y.iloc[tr].nunique()<2: continue
        model=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('logistic',LogisticRegression(class_weight='balanced',max_iter=2000,random_state=SEED))])
        model.fit(X.iloc[tr],y.iloc[tr]); out[te]=model.predict_proba(X.iloc[te])[:,1]
        yt=y.iloc[te].to_numpy(); pt=out[te]; valid=np.isfinite(pt)
        fold_rows.append({'target':target_name,'fold':fold,'group_column':group_column,'test_groups':'|'.join(sorted(groups.iloc[te].unique())),'rows':len(te),'positive_rows':int(yt.sum()),'roc_auc':roc_auc_score(yt[valid],pt[valid]) if len(np.unique(yt[valid]))==2 else np.nan,'average_precision':average_precision_score(yt[valid],pt[valid]) if yt[valid].sum()>0 else np.nan})
    valid=np.isfinite(out)
    summary={'target':target_name,'group_column':group_column,'rows':int(valid.sum()),'positive_rows':int(y[valid].sum()),'groups':int(groups[valid].nunique()),'roc_auc':roc_auc_score(y[valid],out[valid]) if y[valid].nunique()==2 else np.nan,'average_precision':average_precision_score(y[valid],out[valid]) if y[valid].sum()>0 else np.nan,'predictors':'|'.join(keep)}
    pred=d[['source_group','page','y','x','region_id','diagnostic_class']].copy(); pred['target']=target_name; pred['diagnostic_probability']=out
    return summary,pd.DataFrame(fold_rows),pred

summaries=[]; folds=[]; probe_preds=[]
for args in [
 ('PROTECT_vs_RECONSTRUCT','PROTECT',['RECONSTRUCT']),
 ('REMOVE_vs_RECONSTRUCT','REMOVE',['RECONSTRUCT']),
 ('NOT_RECONSTRUCT_vs_RECONSTRUCT','REMOVE',['RECONSTRUCT']),
]:
    # The third probe is replaced below with combined PROTECT+REMOVE positive handling.
    if args[0]=='NOT_RECONSTRUCT_vs_RECONSTRUCT': continue
    for grouping in ['source_group','page']:
        s,f,p=diagnostic_probe(*args,group_column=grouping); summaries.append(s); folds.append(f); probe_preds.append(p)

# Combined NOT_RECONSTRUCT probe.
d=pixels[pixels.diagnostic_class.isin(['PROTECT','REMOVE','RECONSTRUCT'])].copy(); d['binary_class']=np.where(d.diagnostic_class=='RECONSTRUCT','RECONSTRUCT','NOT_RECONSTRUCT')
orig=pixels; pixels_combined=d.rename(columns={'diagnostic_class':'original_diagnostic_class','binary_class':'diagnostic_class'})
pixels_backup=pixels; pixels=pixels_combined
for grouping in ['source_group','page']:
    s,f,p=diagnostic_probe('NOT_RECONSTRUCT_vs_RECONSTRUCT','NOT_RECONSTRUCT',['RECONSTRUCT'],group_column=grouping); summaries.append(s); folds.append(f); probe_preds.append(p)
pixels=pixels_backup

sep_summary=pd.DataFrame(summaries); sep_folds=pd.concat(folds,ignore_index=True) if folds else pd.DataFrame(); sep_predictions=pd.concat(probe_preds,ignore_index=True) if probe_preds else pd.DataFrame()
sep_summary.to_csv(TABLES/'source_group_separability.csv',index=False)
sep_folds.to_csv(TABLES/'separability_fold_results.csv',index=False)
sep_predictions.to_parquet(TABLES/'diagnostic_probe_predictions.parquet',index=False)
display(sep_summary)


## 11. Boundary-conditioning and M3-to-M4 gap analyses


In [ ]:
boundary_cols=['trusted_print_boundary_contact','paper_boundary_contact','edge_boundary_contact','maximum_mask_depth','mean_mask_depth','endpoint_count_near','junction_count_near','m4_overlap_fraction_locked']
boundary_summary=regions.groupby(['source_group','dominant_class'],as_index=False)[boundary_cols+['area','protect_fraction','remove_fraction','reconstruct_fraction','uncertain_fraction']].mean(numeric_only=True)
boundary_summary.to_csv(TABLES/'boundary_conditioning_analysis.csv',index=False)

# Mask-stage distance to oracle, using explicit semantic pixel sets.
gap_rows=[]
for page,z in page_runtime.items():
    m4=z['m4']; authority=z['authority']
    for arm in ['M3']:
        m=z['m3']; inter=int((m&m4).sum()); union=int((m|m4).sum()); xor=int(np.logical_xor(m,m4).sum())
        gap_rows.append({'page':page,'source_group':meta.loc[page,'source_group'],'arm':arm,'pixels':int(m.sum()),'oracle_pixels':int(m4.sum()),'intersection':inter,'union':union,'iou':inter/max(union,1),'xor_pixels':xor,'normalised_xor':xor/max(int(authority.sum()),1),'extra_vs_oracle':int((m&~m4).sum()),'missing_vs_oracle':int((m4&~m).sum())})
gap=pd.DataFrame(gap_rows); gap.to_csv(TABLES/'m3_to_m4_oracle_gap.csv',index=False)
print(gap[['iou','normalised_xor','extra_vs_oracle','missing_vs_oracle']].describe())


## 12. Visual audit queues and contact sheets


In [ ]:
# Region rankings.
queues=[]
def add_queue(name,frame,sort_cols,ascending,max_n=MAX_VISUAL_CASES_PER_QUEUE):
    q=frame.sort_values(sort_cols,ascending=ascending).head(max_n).copy(); q['queue']=name; q['rank']=np.arange(1,len(q)+1); queues.append(q)
add_queue('largest_PROTECT_exposure',regions,['PROTECT','area'],[False,False])
add_queue('largest_REMOVE_contamination',regions,['REMOVE','area'],[False,False])
add_queue('largest_RECONSTRUCT_targets',regions,['RECONSTRUCT','area'],[False,False])
add_queue('most_mixed_regions',regions,['mixed_operational_region','area'],[False,False])
add_queue('most_UNCERTAIN',regions,['UNCERTAIN','area'],[False,False])
review_queue=pd.concat(queues,ignore_index=True)
review_queue.to_csv(TABLES/'visual_audit_queue.csv',index=False)

def crop_bounds(row,shape,margin=45):
    h,w=shape; x0=max(0,int(row.x)-margin); y0=max(0,int(row.y)-margin); x1=min(w,int(row.x+row.w)+margin); y1=min(h,int(row.y+row.h)+margin); return x0,y0,x1,y1

def class_overlay(page,sub):
    z=page_runtime[page]; base=cv2.cvtColor(z['observed'],cv2.COLOR_GRAY2BGR); layer=base.copy(); alpha=np.zeros(z['observed'].shape,np.float32)
    colours={'PROTECT':(255,90,0),'REMOVE':(0,185,70),'RECONSTRUCT':(0,140,255),'UNCERTAIN':(180,0,180)}
    pp=pixels[pixels.page==page]
    for cls,bgr in colours.items():
        q=pp[pp.diagnostic_class==cls]; m=np.zeros(z['observed'].shape,bool); m[q.y.to_numpy(int),q.x.to_numpy(int)]=True; layer[m]=bgr; alpha[m]=.62
    return (base*(1-alpha[...,None])+layer*alpha[...,None]).astype(np.uint8)

for queue,g in review_queue.groupby('queue'):
    folder=VISUAL/queue; folder.mkdir(parents=True,exist_ok=True)
    for _,row in g.iterrows():
        page=row.page; z=page_runtime[page]; x0,y0,x1,y1=crop_bounds(row,z['observed'].shape)
        clean=cv2.imread(str(CLEAN_MAP[page]),0)
        panels=[z['observed'][y0:y1,x0:x1],(z['m3'][y0:y1,x0:x1]*255).astype(np.uint8),(z['m4'][y0:y1,x0:x1]*255).astype(np.uint8),cv2.cvtColor(class_overlay(page,None)[y0:y1,x0:x1],cv2.COLOR_BGR2RGB),clean[y0:y1,x0:x1]]
        titles=['observed','M3','M4 oracle','post-lock classes','clean truth']
        fig,axes=plt.subplots(1,5,figsize=(18,4))
        for ax,im,title in zip(axes,panels,titles): ax.imshow(im,cmap='gray' if im.ndim==2 else None,vmin=0 if im.ndim==2 else None,vmax=255 if im.ndim==2 else None); ax.set_title(title); ax.axis('off')
        fig.suptitle(f'{queue} #{int(row.rank)} | {page} | {row.region_id} | P/R/Rc/U={int(row.PROTECT)}/{int(row.REMOVE)}/{int(row.RECONSTRUCT)}/{int(row.UNCERTAIN)}')
        plt.tight_layout(); plt.savefig(folder/f'{int(row.rank):03d}_{page}_{row.region_id.split(":")[-1]}.png',dpi=160,bbox_inches='tight'); plt.close(fig)
print('Visual queues:',len(review_queue))


## 13. Verdict logic


In [ ]:
# Numerical cut-offs below are diagnostic readiness rules for Notebook 49, not prior project facts.
# They prevent a purely subjective GO decision but must be interpreted with class balance and visual audit.
source_probe=sep_summary[sep_summary.group_column=='source_group'].copy()
protect_row=source_probe[source_probe.target=='PROTECT_vs_RECONSTRUCT']
remove_row=source_probe[source_probe.target=='REMOVE_vs_RECONSTRUCT']
combined_row=source_probe[source_probe.target=='NOT_RECONSTRUCT_vs_RECONSTRUCT']

def acceptable(row):
    if row.empty: return False
    r=row.iloc[0]
    return bool(pd.notna(r.average_precision) and pd.notna(r.roc_auc) and r.roc_auc>=0.65 and r.average_precision>=0.20 and r.groups>=2)

class_counts=pixels.diagnostic_class.value_counts().to_dict()
source_counts=pixels.groupby('diagnostic_class').source_group.nunique().to_dict()
checks={
 'observable_lock_exists':(LOCK/'candidate_lock.json').exists(),
 'truth_loaded_after_lock':True,
 'no_duplicate_pixels':not pixels.duplicated(['page','y','x']).any(),
 'source_groups_at_least_two':pixels.source_group.nunique()>=2,
 'protect_evidence_present':class_counts.get('PROTECT',0)>0 and source_counts.get('PROTECT',0)>=2,
 'remove_evidence_present':class_counts.get('REMOVE',0)>0 and source_counts.get('REMOVE',0)>=2,
 'reconstruct_evidence_present':class_counts.get('RECONSTRUCT',0)>0 and source_counts.get('RECONSTRUCT',0)>=2,
 'protect_separable_source_oof':acceptable(protect_row),
 'remove_separable_source_oof':acceptable(remove_row),
 'combined_separable_source_oof':acceptable(combined_row),
}
if not all([checks['observable_lock_exists'],checks['truth_loaded_after_lock'],checks['no_duplicate_pixels']]):
    verdict='INVALID_RUN'
elif checks['combined_separable_source_oof'] and (checks['protect_separable_source_oof'] or checks['remove_separable_source_oof']):
    verdict='SIGNAL_SUFFICIENT_FOR_M5'
elif checks['protect_separable_source_oof'] or checks['remove_separable_source_oof'] or checks['combined_separable_source_oof']:
    verdict='SIGNAL_PARTIAL_RESEARCH_ONLY'
else:
    verdict='MASK_SIGNAL_INSUFFICIENT'
verdict_record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':verdict,'checks':checks,'class_counts':class_counts,'source_group_counts_by_class':source_counts,'diagnostic_threshold_note':'AUC/AP readiness cut-offs are Notebook 49 commissioning choices, not production gates.','next_action':{'SIGNAL_SUFFICIENT_FOR_M5':'Pre-register Notebook 50 using only the supported target/classes and frozen observable schema.','SIGNAL_PARTIAL_RESEARCH_ONLY':'Narrow the target or source scope before training; do not build a general M5.','MASK_SIGNAL_INSUFFICIENT':'Stop learned mask work on this evidence and reconsider labels/data/reconstruction.','INVALID_RUN':'Correct the first integrity failure and rerun.'}[verdict]}
atomic_json(OUT/'notebook49_verdict.json',verdict_record)
print(json.dumps(verdict_record,indent=2))


## 14. Experiment identity, executive summary and upload package


In [ ]:
environment={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
atomic_json(OUT/'software_environment.json',environment)
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.7-notebook49-oracle-gap-decomposition','development_only':True,'promotion_authority':False,'trained_model':False,'threshold_optimisation':False,'restorer_changed':False,'source_nb45_run':str(SOURCE),'lock_id':LOCK_ID,'label_version':LABEL_VERSION,'pages':int(pixels.page.nunique()),'source_groups':int(pixels.source_group.nunique()),'pixels':int(len(pixels)),'regions':int(len(regions)),'class_counts':pixels.diagnostic_class.value_counts().to_dict(),'constants':{'seed':SEED,'dark_threshold':DARK_THRESHOLD,'print_ink_tolerance':PRINT_INK_TOLERANCE,'registration_uncertainty_radius':REGISTRATION_UNCERTAINTY_RADIUS,'context_ring_radius':CONTEXT_RING_RADIUS,'boundary_band_radius':BOUNDARY_BAND_RADIUS},'software':environment}
atomic_json(OUT/'experiment_identity.json',identity)

lines=[
 'NOTEBOOK 49: RESIDUAL-MASK ORACLE-GAP DECOMPOSITION','',
 f'Run: {OUT}',f'Notebook 45 source: {SOURCE}',f'Observable lock: {LOCK_ID}',
 f'Pages: {pixels.page.nunique()}',f'Source groups: {pixels.source_group.nunique()}',f'M3 pixels: {len(pixels)}',f'Regions: {len(regions)}','',
 'DIAGNOSTIC CLASS COUNTS',*[f'- {k}: {v}' for k,v in pixels.diagnostic_class.value_counts().to_dict().items()],'',
 f'VERDICT: {verdict}',f'Next action: {verdict_record["next_action"]}','','This notebook did not train or freeze M5. Clean truth was opened only after the observable lock.'
]
(OUT/'executive_summary.txt').write_text('\n'.join(lines))

# Curated upload package.
required_upload=[
 OUT/'executive_summary.txt',OUT/'notebook49_verdict.json',OUT/'experiment_identity.json',OUT/'source_record.json',OUT/'software_environment.json',
 LOCK/'candidate_lock.json',LOCK/'feature_schema.json',TABLES/'class_prevalence_by_page.csv',TABLES/'class_prevalence_by_source_group.csv',
 TABLES/'feature_contrasts.csv',TABLES/'source_group_separability.csv',TABLES/'separability_fold_results.csv',TABLES/'boundary_conditioning_analysis.csv',
 TABLES/'m3_to_m4_oracle_gap.csv',TABLES/'visual_audit_queue.csv',AUDIT/'page_array_audit.csv',AUDIT/'source_nb45_inventory.csv'
]
missing=[str(p) for p in required_upload if not p.exists()]
require(not missing,f'Upload package missing required artefacts: {missing}')
for p in required_upload:
    rel=p.relative_to(OUT); dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(p,dst)
# Curated visuals: copy at most ten per queue.
for folder in sorted(VISUAL.iterdir()) if VISUAL.exists() else []:
    if folder.is_dir():
        for p in sorted(folder.glob('*.png'))[:10]:
            dst=UPLOAD/'review'/folder.name/p.name; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(p,dst)
readme='Notebook 49 curated review package. Heavy pixel/region parquet data remain in the run directory and are identified by the observable lock and experiment identity.'
(UPLOAD/'README.txt').write_text(readme)
upload_rows=[]
for p in sorted(UPLOAD.rglob('*')):
    if p.is_file(): upload_rows.append({'relative_path':str(p.relative_to(UPLOAD)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(upload_rows).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'upload_file_list.txt').write_text('\n'.join(x['relative_path'] for x in upload_rows))
zip_path=UPLOAD/'notebook49_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file() and p!=zip_path: zf.write(p,p.relative_to(UPLOAD))
require(zip_path.exists() and zip_path.stat().st_size>0,'Upload bundle creation failed.')
print('\n'.join(lines)); print('\nUpload bundle:',zip_path)


## Outputs to inspect first

1. `executive_summary.txt`
2. `notebook49_verdict.json`
3. `04_tables/class_prevalence_by_page.csv`
4. `04_tables/class_prevalence_by_source_group.csv`
5. `04_tables/source_group_separability.csv`
6. `04_tables/feature_contrasts.csv`
7. `04_tables/boundary_conditioning_analysis.csv`
8. `04_tables/m3_to_m4_oracle_gap.csv`
9. `05_visual_audit/`
10. `upload/notebook49_upload_bundle.zip`

A `SIGNAL_SUFFICIENT_FOR_M5` verdict permits a separate Notebook 50 pre-registration. It does not select an architecture, threshold or production policy.
